### PageIndex + Ollama Legal Bot

This notebook is a demonstration of the retrieval and ranking capabilities of Vectorless RAG system using PageIndex + Ollama local model.

Code by Saish Shetty. Dated 10/06/2026

#### Step 1: Install all dependencies

In [1]:
#pip install -q --upgrade openai-agents openai pageindex python_dotenv

#### Step 2: Load environment variable and connect to clients

In [1]:
import os
from dotenv import load_dotenv
from pageindex import PageIndexClient

load_dotenv()

## Fetch PageIndex API Key
## Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv("pi-test-key", None)

## Create PageIndexClient
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

#### Step 3: Upload knowledge documents

In [ ]:
# ## We will test with the Constitution of India - which is a perfect example of a document with hierarchical structure
# import os, requests, time

# pdf_path = os.path.join("document", "constitutionofindiaacts.pdf")

# # if not os.path.exists(pdf_path):
# #     with open(pdf_path, "wb") as f:
# #         f.write(requests.get(pdf_url).content)

# doc_id = pi_client.submit_document(pdf_path)["doc_id"]
# print(f"Submitted: {doc_id}")

# # Wait for processing
# while pi_client.get_document(doc_id)["status"] != "completed":
#     time.sleep(5)
# print(f"Ready: {pi_client.get_document(doc_id)['name']}")

In [ ]:
# Submitted: pi-cmq87lbk200lm01qo4f0xqqkl
# Ready: constitutionofindiaacts.pdf

#### Step 4: Analyze Document Tree Structure

In [ ]:
# tree_result = pi_client.get_tree(doc_id)["result"]

#### Step 5: Use Tree+Query to get coherent LLM Responses

In [2]:
from openai import AsyncOpenAI
from agents import Agent, Runner, ModelSettings, set_tracing_disabled
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStreamableHttp

set_tracing_disabled(True)

## Create Ollama Model client using OpenAI model signature
MODEL = "qwen3.5"
client = AsyncOpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored for local use
)

async def ask(question, show_tool_results=False):
    try:
        async with MCPServerStreamableHttp(
            name="pageindex",
            params={"url": "https://api.pageindex.ai/mcp", "headers": {"Authorization": f"Bearer {PAGEINDEX_API_KEY}"}},
            cache_tools_list=True,
        ) as server:
            agent = Agent(
                name="PageIndex RAG Agent",
                model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
                # instructions="Use the PageIndex MCP tools to answer questions about the user's documents.",
                instructions="Use the PageIndex MCP tools to answer questions about the user's documents.",
                mcp_servers=[server],
                model_settings=ModelSettings(reasoning={"effort": "medium", "summary": "auto"}),
            )
            result = Runner.run_streamed(agent, question)
            event_count = 0
            tool_call_count = 0
            async for event in result.stream_events():
                event_count += 1
                
                # Log all run_item_stream_event with full details
                if event.type == "run_item_stream_event":
                    tool_call_count += 1
                    print(f"\n[TOOL_RESULT #{tool_call_count}]", flush=True)
                    print(f"  Event #{event_count}: {event.type}", flush=True)
                    print(f"  Item type: {event.item.type}", flush=True)
                    if hasattr(event.item, 'output'):
                        output = event.item.output
                        print(f"  Output length: {len(str(output))}", flush=True)
                        print(f"  Output preview: {str(output)[:500]}", flush=True)
                    if hasattr(event.item, 'error'):
                        print(f"  Error: {event.item.error}", flush=True)
                    print(f"  Full item: {event.item}", flush=True)
                    continue
                
                if event.type == "raw_response_event":
                    data = event.data
                    if data.type == "response.output_item.added":
                        item = data.item
                        if item.type == "reasoning":
                            print(f"\n[{item.type}] ", end="", flush=True)
                        elif item.type == "function_call":
                            print(f"\n[{item.type}] {item.name} ", end="", flush=True)
                    elif data.type in ("response.reasoning_summary_text.delta", "response.reasoning_text.delta", "response.output_text.delta", "response.function_call_arguments.delta"):
                        print(data.delta, end="", flush=True)
                    elif data.type == "response.output_item.done" and data.item.type == "function_call":
                        print()
            
            print(f"\n[STREAM_END] Total events: {event_count}, Tool results: {tool_call_count}", flush=True)
            
    except Exception as e:
        print(f"\n[ERROR] Exception occurred: {type(e).__name__}: {e}", flush=True)
        import traceback
        traceback.print_exc()

In [3]:
## Ask Query
await ask("What is the provision for women in the 131st amendment bill?")


[reasoning] The user is asking about a specific provision related to women in a "131st amendment bill". This seems like they want information from legal or policy documents. I should start by browsing the document library to see what's available, and then search for documents that might contain information about this amendment bill.

Let me first get an overview of the folder structure to understand what type of documents are available in the library.
[function_call] browse_documents {"folder_id":"root","recursive":false,"sort":"relevance","query":"131st amendment bill women provision"}
[TOOL_RESULT #1]
  Event #178: run_item_stream_event
  Item type: reasoning_item
  Full item: ReasoningItem(agent=Agent(name='PageIndex RAG Agent', handoff_description=None, tools=[], mcp_servers=[<agents.mcp.server.MCPServerStreamableHttp object at 0x0000027ECBD5F910>], mcp_config={}, instructions="Use the PageIndex MCP tools to answer questions about the user's documents.", prompt=None, handoffs=[], 